# Construction of a Disjoint 500-Sample MMMU-Pro Held-Out Test Cohort

**Purpose.** This notebook constructs a **500-sample independent held-out cohort** from the **MMMU-Pro Standard (10 options)** benchmark for final evaluation of an adaptive prompt-routing system.

The cohort is designed to satisfy four methodological requirements:

1. **No ID overlap with the 300-sample development cohort.**
2. **No ID overlap with the demonstration / CoT bank.**
3. **No exact normalized question-text overlap** with either the development cohort (when recoverable from MMMU-Pro) or the demonstration bank.
4. **Balanced subject coverage with controlled difficulty composition**, while sampling strictly without replacement and without using answer labels or model-performance results.

The 300 previously evaluated samples are treated as the **development/routing set**. The 500 samples produced here are held out and must not be used to revise prompt-selection rules after their evaluation.

---

### Dataset version

This notebook pins the Hugging Face dataset repository to:

- Repository: `MMMU/MMMU_Pro`
- Configuration: `standard (10 options)`
- Split: `test`
- Revision: `563f3e84bb3b90893083a1f039cfa13077f2302b`

At this revision, the dataset card reports **1,730 examples** for the Standard (10 options) split. The dataset card also records benchmark corrections through **10 July 2026**.

**Primary sources**

- MMMU-Pro dataset: https://huggingface.co/datasets/MMMU/MMMU_Pro
- MMMU-Pro paper: https://arxiv.org/abs/2409.02813

> **Reproducibility principle:** all sampling decisions are controlled by a fixed random seed, the dataset revision is pinned, all exclusion files are hashed, and the final cohort is accompanied by a machine-readable audit manifest.

## 1. Sampling protocol

Let \(D\) denote the pinned MMMU-Pro Standard (10 options) dataset. Let \(E_{\mathrm{dev}}\) be the set of IDs from the previously used 300-sample development cohort and \(E_{\mathrm{demo}}\) the set of IDs from the demonstration/CoT bank.

The eligible candidate pool is constructed as

\[
D_{\mathrm{eligible}}
=
D \setminus
\left(
E_{\mathrm{dev}}
\cup
E_{\mathrm{demo}}
\cup
E_{\mathrm{exact\text{-}text}}
\right),
\]

where \(E_{\mathrm{exact\text{-}text}}\) is a conservative secondary contamination guard based only on **exact equality after Unicode and whitespace normalization**. No fuzzy matching, embedding similarity, answer correctness, model score, prompt result, or other performance-dependent criterion is used in cohort construction.

### Stratification

The target cohort size is \(N=500\) across the 30 MMMU-Pro subjects.

- Subject allocation is as even as possible. With sufficient availability, 20 subjects receive 17 samples and 10 receive 16 samples.
- Within each subject, selection targets the same difficulty proportion used in the development design:
  - Easy: 30%
  - Medium: 40%
  - Hard: 30%
- If a subject×difficulty stratum cannot satisfy its ideal quota after exclusions, the notebook computes the closest feasible allocation for that subject and records the deviation.
- Sampling is without replacement.

This procedure keeps the held-out cohort representative across subjects while avoiding any selection based on downstream model behavior.

## 2. Environment and imports

Kaggle commonly includes the required packages. If `datasets` or `huggingface_hub` is missing, run:

```bash
pip install "datasets>=3.0,<5.0" "huggingface_hub>=0.25,<2.0"
```

For the first uncached download from Hugging Face, Kaggle Internet access must be enabled.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import json
import platform
import random
import re
import sys
import unicodedata
import zipfile

import numpy as np
import pandas as pd

try:
    from datasets import load_dataset
except ImportError as exc:
    raise ImportError(
        "The 'datasets' package is required. Install it with: "
        "pip install 'datasets>=3.0,<5.0'"
    ) from exc

print(f"Python   : {sys.version.split()[0]}")
print(f"NumPy    : {np.__version__}")
print(f"pandas   : {pd.__version__}")

try:
    print(f"datasets : {version('datasets')}")
except PackageNotFoundError:
    pass

Python   : 3.12.13
NumPy    : 2.0.2
pandas   : 2.3.3
datasets : 5.0.0


## 3. Configuration

The two exclusion files may be placed anywhere under `/kaggle/input`. The resolver searches by filename pattern and refuses to guess if multiple matching files are found. If that happens, set `SELECTED_IDS_PATH` or `DEMO_CSV_PATH` explicitly.

The exclusion inputs are expected to contain:

- the original **300 development IDs** in a text file, one ID per line;
- the demonstration bank in CSV format with at least `question_id` and `question`.

In [2]:
SEED = 42
TARGET_N = 500
EXPECTED_DEVELOPMENT_N = 300
EXPECTED_MMMU_PRO_N = 1730
EXPECTED_SUBJECT_N = 30

random.seed(SEED)
np.random.seed(SEED)

DATASET_REPO = "MMMU/MMMU_Pro"
DATASET_CONFIG = "standard (10 options)"
DATASET_SPLIT = "test"
DATASET_REVISION = "563f3e84bb3b90893083a1f039cfa13077f2302b"

DIFFICULTY_ORDER = ("Easy", "Medium", "Hard")
TARGET_DIFFICULTY_PROPORTIONS = {
    "Easy": 0.30,
    "Medium": 0.40,
    "Hard": 0.30,
}

# Set either variable to Path("...") to override automatic discovery.
SELECTED_IDS_PATH = None
DEMO_CSV_PATH = None

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

SELECTED_IDS_PATTERNS = (
    "selected_ids.txt",
    "selected_ids*.txt",
)
DEMO_CSV_PATTERNS = (
    "cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv",
    "cot_pipeline_clean_merged_KEEP_only_grouped_by_subject*.csv",
)

OUTPUT_DIR = Path("/kaggle/working/mmmu_pro_heldout_500")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /kaggle/working/mmmu_pro_heldout_500


## 4. Resolve the exclusion files

Automatic discovery is deliberately strict: if more than one matching file exists, the notebook stops and prints the candidates instead of silently choosing one.

In [3]:
def resolve_input_file(
    explicit_path: Path | str | None,
    patterns: tuple[str, ...],
    root: Path,
    label: str,
) -> Path:
    if explicit_path is not None:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f"{label} not found: {path}")
        return path.resolve()

    if not root.exists():
        raise FileNotFoundError(
            f"Kaggle input root does not exist: {root}. "
            "Attach the input files or set an explicit path."
        )

    matches = set()
    for pattern in patterns:
        matches.update(p.resolve() for p in root.rglob(pattern) if p.is_file())

    matches = sorted(matches)

    if not matches:
        raise FileNotFoundError(
            f"Could not locate {label} under {root}. "
            f"Searched patterns: {patterns}. "
            "Set its explicit path in the configuration cell."
        )

    if len(matches) > 1:
        formatted = "\n".join(f"  - {p}" for p in matches)
        raise RuntimeError(
            f"Multiple candidates found for {label}; refusing to guess.\n"
            f"{formatted}\n"
            "Set the corresponding explicit path in the configuration cell."
        )

    return matches[0]


SELECTED_IDS_PATH = resolve_input_file(
    SELECTED_IDS_PATH,
    SELECTED_IDS_PATTERNS,
    KAGGLE_INPUT_ROOT,
    "development ID file",
)
DEMO_CSV_PATH = resolve_input_file(
    DEMO_CSV_PATH,
    DEMO_CSV_PATTERNS,
    KAGGLE_INPUT_ROOT,
    "demonstration CSV",
)

print("Development IDs :", SELECTED_IDS_PATH)
print("Demonstration CSV:", DEMO_CSV_PATH)

Development IDs : /kaggle/input/datasets/jingilifteyna/300-mmmu-pro/selected_ids.txt
Demonstration CSV: /kaggle/input/datasets/jingilifteyna/pooling/cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv


## 5. Load and validate exclusion sources

Only the fields needed for contamination control are read from the demonstration CSV. This avoids coupling cohort selection to any generated CoT, validator result, or model output contained elsewhere in that file.

In [4]:
def normalize_id(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_question(value: object) -> str:
    # Conservative exact-text normalization only:
    # Unicode NFKC -> whitespace collapse -> case folding.
    # This is NOT fuzzy or semantic matching.
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip().casefold()
    return text


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


with SELECTED_IDS_PATH.open("r", encoding="utf-8-sig") as f:
    raw_development_ids = [normalize_id(line) for line in f if line.strip()]

if not raw_development_ids:
    raise ValueError("The development ID file is empty.")

duplicate_development_ids = [
    item for item, count in Counter(raw_development_ids).items() if count > 1
]
if duplicate_development_ids:
    raise ValueError(
        "Duplicate IDs found in the development file. "
        f"Examples: {duplicate_development_ids[:10]}"
    )

development_ids = set(raw_development_ids)

if len(development_ids) != EXPECTED_DEVELOPMENT_N:
    raise ValueError(
        f"Expected exactly {EXPECTED_DEVELOPMENT_N} development IDs, "
        f"but found {len(development_ids)}."
    )

required_demo_columns = {"question_id", "question"}
demo_df = pd.read_csv(
    DEMO_CSV_PATH,
    usecols=lambda c: c in required_demo_columns,
)

missing_demo_columns = required_demo_columns - set(demo_df.columns)
if missing_demo_columns:
    raise ValueError(
        f"Demonstration CSV is missing required columns: "
        f"{sorted(missing_demo_columns)}"
    )

demo_df["question_id_norm"] = demo_df["question_id"].map(normalize_id)
demo_df["question_norm"] = demo_df["question"].map(normalize_question)

if (demo_df["question_id_norm"] == "").any():
    raise ValueError("The demonstration CSV contains missing/empty question IDs.")

duplicate_demo_ids = (
    demo_df.loc[
        demo_df["question_id_norm"].duplicated(keep=False),
        "question_id_norm",
    ]
    .drop_duplicates()
    .tolist()
)
if duplicate_demo_ids:
    raise ValueError(
        "Duplicate question_id values found in the demonstration CSV. "
        f"Examples: {duplicate_demo_ids[:10]}"
    )

demonstration_ids = set(demo_df["question_id_norm"])
demonstration_questions = {q for q in demo_df["question_norm"] if q}

print("=" * 78)
print("EXCLUSION SOURCE AUDIT")
print("=" * 78)
print(f"Development IDs (unique)           : {len(development_ids):,}")
print(f"Demonstration IDs (unique)         : {len(demonstration_ids):,}")
print(
    "Exact ID overlap between sources   : "
    f"{len(development_ids & demonstration_ids):,}"
)
print(
    "Demo questions available for guard : "
    f"{len(demonstration_questions):,}"
)
print(f"Development file SHA-256           : {sha256_file(SELECTED_IDS_PATH)}")
print(f"Demonstration CSV SHA-256          : {sha256_file(DEMO_CSV_PATH)}")

EXCLUSION SOURCE AUDIT
Development IDs (unique)           : 300
Demonstration IDs (unique)         : 405
Exact ID overlap between sources   : 0
Demo questions available for guard : 389
Development file SHA-256           : db7ec6dca5dff71d8ea8551a08c448c4314e153509ce9c78d2e8c652011a5dc3
Demonstration CSV SHA-256          : 2c55994efcbaf2d1e21e43f256a70e3e10859f6f20eafb68c41a48d9ba2666ca


## 6. Load the pinned MMMU-Pro dataset

Only non-image metadata required for cohort construction are materialized into pandas: `id`, `question`, `subject`, and `topic_difficulty`. **Ground-truth answers are intentionally not used in selection.**

The image columns remain in the Hugging Face dataset object for later model evaluation, but they play no role in choosing the 500 held-out instances.

In [5]:
print("Loading pinned MMMU-Pro dataset...")

ds = load_dataset(
    DATASET_REPO,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
    revision=DATASET_REVISION,
)

if len(ds) != EXPECTED_MMMU_PRO_N:
    raise RuntimeError(
        f"Dataset-size mismatch: expected {EXPECTED_MMMU_PRO_N}, got {len(ds)}. "
        "Do not continue until the dataset revision/configuration is verified."
    )

required_columns = {"id", "question", "subject", "topic_difficulty"}
missing_columns = required_columns - set(ds.column_names)
if missing_columns:
    raise RuntimeError(
        f"MMMU-Pro is missing required fields: {sorted(missing_columns)}"
    )

metadata = pd.DataFrame({
    "dataset_index": np.arange(len(ds), dtype=np.int64),
    "id": ds["id"],
    "question": ds["question"],
    "subject": ds["subject"],
    "topic_difficulty": ds["topic_difficulty"],
})

metadata["id"] = metadata["id"].map(normalize_id)
metadata["question_norm"] = metadata["question"].map(normalize_question)
metadata["subject"] = metadata["subject"].astype(str).str.strip()
metadata["topic_difficulty"] = (
    metadata["topic_difficulty"]
    .astype(str)
    .str.strip()
    .str.title()
)

if metadata["id"].eq("").any():
    raise RuntimeError("MMMU-Pro contains empty IDs.")

if not metadata["id"].is_unique:
    duplicated = (
        metadata.loc[metadata["id"].duplicated(keep=False), "id"]
        .drop_duplicates()
        .tolist()
    )
    raise RuntimeError(
        f"MMMU-Pro contains duplicate IDs. Examples: {duplicated[:10]}"
    )

observed_subjects = sorted(metadata["subject"].unique())
if len(observed_subjects) != EXPECTED_SUBJECT_N:
    raise RuntimeError(
        f"Expected {EXPECTED_SUBJECT_N} subjects, found {len(observed_subjects)}."
    )

observed_difficulties = set(metadata["topic_difficulty"].unique())
unexpected_difficulties = observed_difficulties - set(DIFFICULTY_ORDER)
if unexpected_difficulties:
    raise RuntimeError(
        f"Unexpected difficulty labels: {sorted(unexpected_difficulties)}"
    )

print(f"MMMU-Pro examples : {len(metadata):,}")
print(f"Subjects          : {len(observed_subjects)}")
print("Difficulty counts:")
display(
    metadata["topic_difficulty"]
    .value_counts()
    .reindex(DIFFICULTY_ORDER, fill_value=0)
    .rename("count")
    .to_frame()
)

Loading pinned MMMU-Pro dataset...


README.md: 0.00B [00:00, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]

MMMU-Pro examples : 1,730
Subjects          : 30
Difficulty counts:


,count
topic_difficulty,
Easy,528
Medium,801
Hard,401


## 7. Construct the contamination mask

The primary exclusion rule is exact ID matching. A second, conservative safeguard excludes exact normalized question-text matches.

For the 300 development items, their question text is first recovered directly from the pinned MMMU-Pro revision. Therefore, an alternative row with a different ID but exactly the same normalized question would also be excluded.

The audit below reports each exclusion reason separately; a row may satisfy more than one reason.

In [6]:
mmmu_ids = set(metadata["id"])

missing_development_ids = sorted(development_ids - mmmu_ids)
if missing_development_ids:
    raise RuntimeError(
        "Some development IDs are absent from the pinned MMMU-Pro revision. "
        "This usually indicates a dataset-version mismatch. "
        f"Missing count: {len(missing_development_ids)}; "
        f"examples: {missing_development_ids[:10]}"
    )

development_questions = set(
    metadata.loc[
        metadata["id"].isin(development_ids)
        & metadata["question_norm"].ne(""),
        "question_norm",
    ]
)

metadata["exclude_dev_id"] = metadata["id"].isin(development_ids)
metadata["exclude_demo_id"] = metadata["id"].isin(demonstration_ids)
metadata["exclude_dev_exact_text"] = metadata["question_norm"].isin(
    development_questions
)
metadata["exclude_demo_exact_text"] = metadata["question_norm"].isin(
    demonstration_questions
)

exclusion_columns = [
    "exclude_dev_id",
    "exclude_demo_id",
    "exclude_dev_exact_text",
    "exclude_demo_exact_text",
]

metadata["excluded_any"] = metadata[exclusion_columns].any(axis=1)

excluded = metadata.loc[metadata["excluded_any"]].copy()
candidates = metadata.loc[~metadata["excluded_any"]].copy()

audit_counts = pd.Series({
    "MMMU-Pro total": len(metadata),
    "Development IDs found in MMMU-Pro": int(metadata["exclude_dev_id"].sum()),
    "Demo IDs found in MMMU-Pro": int(metadata["exclude_demo_id"].sum()),
    "Rows matching development exact text": int(
        metadata["exclude_dev_exact_text"].sum()
    ),
    "Rows matching demo exact text": int(
        metadata["exclude_demo_exact_text"].sum()
    ),
    "Unique rows excluded by any rule": len(excluded),
    "Eligible candidate rows": len(candidates),
})

display(audit_counts.rename("count").to_frame())

if len(candidates) < TARGET_N:
    raise RuntimeError(
        f"Only {len(candidates)} eligible candidates remain; "
        f"{TARGET_N} are required."
    )

candidate_subject_counts = (
    candidates["subject"]
    .value_counts()
    .sort_index()
)

print("\nEligible candidate range per subject:")
print(
    f"min={candidate_subject_counts.min()}, "
    f"max={candidate_subject_counts.max()}, "
    f"median={candidate_subject_counts.median():.1f}"
)

,count
MMMU-Pro total,1730
Development IDs found in MMMU-Pro,300
Demo IDs found in MMMU-Pro,68
Rows matching development exact text,335
Rows matching demo exact text,80
Unique rows excluded by any rule,410
Eligible candidate rows,1320



Eligible candidate range per subject:
min=30, max=50, median=46.0


## 8. Define the balanced allocation procedure

Two deterministic allocation functions are used.

### 8.1 Subject allocation

A seeded randomized round-robin allocator distributes 500 slots across subjects while respecting post-exclusion capacity. When all subjects have sufficient capacity, this produces the maximally balanced \(16/17\) allocation.

The randomization affects **which** 20 subjects receive the seventeenth item; it does not affect the number of subjects represented or the total cohort size.

### 8.2 Difficulty allocation

For each subject, the notebook searches all feasible `(Easy, Medium, Hard)` integer allocations summing to that subject's target and chooses the allocation minimizing squared deviation from the desired 30:40:30 proportions. This makes the fallback explicit and auditable if a stratum has limited availability.

In [7]:
def balanced_subject_allocation(
    available_by_subject: pd.Series,
    total_n: int,
    seed: int,
) -> dict[str, int]:
    # Allocate slots as evenly as possible across subjects while
    # respecting each subject's post-exclusion capacity.
    if total_n > int(available_by_subject.sum()):
        raise ValueError("Requested total exceeds the candidate pool.")

    subjects = np.array(sorted(available_by_subject.index), dtype=object)
    rng = np.random.default_rng(seed)
    order = subjects.copy()
    rng.shuffle(order)

    targets = {str(subject): 0 for subject in subjects}
    assigned = 0

    while assigned < total_n:
        progressed = False

        for subject_obj in order:
            subject = str(subject_obj)

            if assigned >= total_n:
                break

            if targets[subject] < int(available_by_subject.loc[subject]):
                targets[subject] += 1
                assigned += 1
                progressed = True

        if not progressed:
            raise RuntimeError(
                "Balanced allocation stalled before reaching the requested size."
            )

    return targets


def closest_feasible_difficulty_allocation(
    target_n: int,
    availability: dict[str, int],
    proportions: dict[str, float],
) -> dict[str, int]:
    # Exhaustively choose the feasible integer allocation closest
    # to the requested Easy/Medium/Hard proportions.
    if sum(availability.get(k, 0) for k in DIFFICULTY_ORDER) < target_n:
        raise ValueError(
            f"Insufficient candidates for target_n={target_n}: {availability}"
        )

    ideal = {
        k: target_n * proportions[k]
        for k in DIFFICULTY_ORDER
    }

    best_key = None
    best_counts = None

    max_easy = min(availability.get("Easy", 0), target_n)
    max_medium = min(availability.get("Medium", 0), target_n)

    for easy_n in range(max_easy + 1):
        for medium_n in range(max_medium + 1):
            hard_n = target_n - easy_n - medium_n

            if hard_n < 0:
                continue
            if hard_n > availability.get("Hard", 0):
                continue

            counts = {
                "Easy": easy_n,
                "Medium": medium_n,
                "Hard": hard_n,
            }

            squared_error = sum(
                (counts[k] - ideal[k]) ** 2
                for k in DIFFICULTY_ORDER
            )
            max_abs_error = max(
                abs(counts[k] - ideal[k])
                for k in DIFFICULTY_ORDER
            )

            key = (
                round(squared_error, 12),
                round(max_abs_error, 12),
                tuple(counts[k] for k in DIFFICULTY_ORDER),
            )

            if best_key is None or key < best_key:
                best_key = key
                best_counts = counts

    if best_counts is None:
        raise RuntimeError(
            f"No feasible difficulty allocation for target {target_n} "
            f"with availability {availability}."
        )

    return best_counts

## 9. Allocate and sample the 500 held-out instances

Sampling occurs only after all exclusions have been finalized. A single NumPy random generator seeded with `SEED` is used for row selection and final evaluation-order shuffling.

The output order is randomized once so that downstream model evaluation is not clustered by subject.

In [8]:
available_by_subject = (
    candidates["subject"]
    .value_counts()
    .sort_index()
)

subject_targets = balanced_subject_allocation(
    available_by_subject=available_by_subject,
    total_n=TARGET_N,
    seed=SEED,
)

if sum(subject_targets.values()) != TARGET_N:
    raise RuntimeError("Subject target allocation does not sum to TARGET_N.")

rng = np.random.default_rng(SEED)

selected_indices = []
allocation_records = []

for subject in sorted(subject_targets):
    subject_pool = candidates.loc[candidates["subject"] == subject]
    subject_target = subject_targets[subject]

    availability = {
        difficulty: int(
            (subject_pool["topic_difficulty"] == difficulty).sum()
        )
        for difficulty in DIFFICULTY_ORDER
    }

    quota = closest_feasible_difficulty_allocation(
        target_n=subject_target,
        availability=availability,
        proportions=TARGET_DIFFICULTY_PROPORTIONS,
    )

    for difficulty in DIFFICULTY_ORDER:
        stratum = subject_pool.loc[
            subject_pool["topic_difficulty"] == difficulty
        ]
        n_take = quota[difficulty]

        if n_take == 0:
            continue

        chosen = rng.choice(
            stratum.index.to_numpy(),
            size=n_take,
            replace=False,
        )
        selected_indices.extend(chosen.tolist())

    allocation_records.append({
        "subject": subject,
        "eligible_total": len(subject_pool),
        "target_total": subject_target,
        "available_easy": availability["Easy"],
        "available_medium": availability["Medium"],
        "available_hard": availability["Hard"],
        "selected_easy": quota["Easy"],
        "selected_medium": quota["Medium"],
        "selected_hard": quota["Hard"],
    })

if len(selected_indices) != TARGET_N:
    raise RuntimeError(
        f"Sampling produced {len(selected_indices)} rows instead of {TARGET_N}."
    )

selected = candidates.loc[selected_indices].copy()

permutation = rng.permutation(len(selected))
selected = selected.iloc[permutation].reset_index(drop=True)
selected.insert(0, "selection_order", np.arange(1, len(selected) + 1))

allocation_df = pd.DataFrame(allocation_records)

print(f"Selected rows: {len(selected):,}")
display(allocation_df)

Selected rows: 500


,subject,eligible_total,target_total,available_easy,available_medium,available_hard,selected_easy,selected_medium,selected_hard
0,Accounting,44,17,16,24,4,6,7,4
1,Agriculture,49,16,14,2,33,7,2,7
2,Architecture_and_Engineering,50,16,4,19,27,4,7,5
3,Art,30,17,18,10,2,7,8,2
4,Art_Theory,41,16,23,15,3,6,7,3
5,Basic_Medical_Science,35,17,11,20,4,6,7,4
6,Biology,46,17,19,24,3,6,8,3
7,Chemistry,48,17,13,19,16,5,7,5
8,Clinical_Medicine,42,16,6,29,7,5,6,5
9,Computer_Science,46,17,16,21,9,5,7,5


## 10. Leakage and integrity tests

These assertions are intentionally strict. The notebook terminates if any selected row overlaps the development set or demonstration bank by ID or exact normalized question text.

The test also verifies uniqueness and confirms that every selected row came from the eligible candidate pool.

In [9]:
selected_ids = set(selected["id"])
selected_questions = {
    q for q in selected["question_norm"] if q
}

checks = {
    "exactly_500_rows": len(selected) == TARGET_N,
    "500_unique_ids": selected["id"].nunique() == TARGET_N,
    "no_development_id_overlap": len(selected_ids & development_ids) == 0,
    "no_demo_id_overlap": len(selected_ids & demonstration_ids) == 0,
    "no_development_exact_text_overlap": (
        len(selected_questions & development_questions) == 0
    ),
    "no_demo_exact_text_overlap": (
        len(selected_questions & demonstration_questions) == 0
    ),
    "all_rows_from_candidate_pool": set(
        selected["dataset_index"]
    ).issubset(set(candidates["dataset_index"])),
    "all_30_subjects_represented": (
        selected["subject"].nunique() == EXPECTED_SUBJECT_N
    ),
}

check_df = pd.DataFrame(
    [
        {"check": name, "passed": bool(value)}
        for name, value in checks.items()
    ]
)

display(check_df)

failed_checks = check_df.loc[~check_df["passed"], "check"].tolist()
if failed_checks:
    raise AssertionError(f"Integrity checks failed: {failed_checks}")

print("All leakage and integrity checks PASSED.")

,check,passed
0,exactly_500_rows,True
1,500_unique_ids,True
2,no_development_id_overlap,True
3,no_demo_id_overlap,True
4,no_development_exact_text_overlap,True
5,no_demo_exact_text_overlap,True
6,all_rows_from_candidate_pool,True
7,all_30_subjects_represented,True


All leakage and integrity checks PASSED.


## 11. Distribution audit

The following tables document the final subject and difficulty composition. Under unconstrained availability, the intended design is:

- 20 subjects × 17 samples;
- 10 subjects × 16 samples;
- approximately 150 Easy, 200 Medium, and 150 Hard instances overall.

If exclusions make a particular difficulty stratum too small, the exact realized allocation is shown rather than hidden.

In [10]:
subject_distribution = (
    selected["subject"]
    .value_counts()
    .sort_index()
    .rename("n")
    .to_frame()
)

difficulty_distribution = (
    selected["topic_difficulty"]
    .value_counts()
    .reindex(DIFFICULTY_ORDER, fill_value=0)
    .rename("n")
    .to_frame()
)
difficulty_distribution["proportion"] = (
    difficulty_distribution["n"] / TARGET_N
)
difficulty_distribution["target_proportion"] = pd.Series(
    TARGET_DIFFICULTY_PROPORTIONS
)
difficulty_distribution["absolute_deviation"] = (
    difficulty_distribution["proportion"]
    - difficulty_distribution["target_proportion"]
).abs()

subject_by_difficulty = pd.crosstab(
    selected["subject"],
    selected["topic_difficulty"],
).reindex(
    columns=DIFFICULTY_ORDER,
    fill_value=0,
)
subject_by_difficulty["Total"] = subject_by_difficulty.sum(axis=1)

print("Subject counts:")
display(subject_distribution)

print("Overall difficulty distribution:")
display(difficulty_distribution)

print("Subject × difficulty distribution:")
display(subject_by_difficulty)

print(
    "Subject count range:",
    int(subject_distribution["n"].min()),
    "to",
    int(subject_distribution["n"].max()),
)

Subject counts:


,n
subject,
Accounting,17
Agriculture,16
Architecture_and_Engineering,16
Art,17
Art_Theory,16
Basic_Medical_Science,17
Biology,17
Chemistry,17
Clinical_Medicine,16


Overall difficulty distribution:


,n,proportion,target_proportion,absolute_deviation
topic_difficulty,,,,
Easy,166,0.332,0.3,0.032
Medium,209,0.418,0.4,0.018
Hard,125,0.250,0.3,0.050


Subject × difficulty distribution:


topic_difficulty,Easy,Medium,Hard,Total
subject,,,,
Accounting,6,7,4,17
Agriculture,7,2,7,16
Architecture_and_Engineering,4,7,5,16
Art,7,8,2,17
Art_Theory,6,7,3,16
Basic_Medical_Science,6,7,4,17
Biology,6,8,3,17
Chemistry,5,7,5,17
Clinical_Medicine,5,6,5,16


Subject count range: 16 to 17


## 12. Candidate-availability audit

This table documents the eligible pool **after all contamination exclusions**. It is useful when explaining any deviation from the ideal within-subject 30:40:30 difficulty target.

In [11]:
candidate_by_subject_difficulty = pd.crosstab(
    candidates["subject"],
    candidates["topic_difficulty"],
).reindex(
    columns=DIFFICULTY_ORDER,
    fill_value=0,
)
candidate_by_subject_difficulty["Total"] = (
    candidate_by_subject_difficulty.sum(axis=1)
)

display(candidate_by_subject_difficulty)

topic_difficulty,Easy,Medium,Hard,Total
subject,,,,
Accounting,16,24,4,44
Agriculture,14,2,33,49
Architecture_and_Engineering,4,19,27,50
Art,18,10,2,30
Art_Theory,23,15,3,41
Basic_Medical_Science,11,20,4,35
Biology,19,24,3,46
Chemistry,13,19,16,48
Clinical_Medicine,6,29,7,42


## 13. Export the frozen cohort and reproducibility artifacts

Outputs:

- `selected_500_ids.txt` — fixed randomized evaluation order;
- `selected_500_ids_sorted.txt` — canonical sorted ID list;
- `selected_500_metadata.csv` — selection order, MMMU-Pro index, subject, difficulty;
- `subject_difficulty_distribution.csv`;
- `candidate_subject_difficulty_distribution.csv`;
- `exclusion_audit.csv`;
- `allocation_audit.csv`;
- `selection_manifest.json`;
- `SAMPLING_PROTOCOL.md`;
- one ZIP bundle containing all artifacts.

The manifest records input hashes, cohort hashes, software versions, dataset revision, exclusion counts, and all integrity checks.

In [12]:
OUTPUT_IDS = OUTPUT_DIR / "selected_500_ids.txt"
OUTPUT_IDS_SORTED = OUTPUT_DIR / "selected_500_ids_sorted.txt"
OUTPUT_METADATA = OUTPUT_DIR / "selected_500_metadata.csv"
OUTPUT_DISTRIBUTION = OUTPUT_DIR / "subject_difficulty_distribution.csv"
OUTPUT_CANDIDATES = (
    OUTPUT_DIR / "candidate_subject_difficulty_distribution.csv"
)
OUTPUT_EXCLUSION_AUDIT = OUTPUT_DIR / "exclusion_audit.csv"
OUTPUT_ALLOCATION_AUDIT = OUTPUT_DIR / "allocation_audit.csv"
OUTPUT_MANIFEST = OUTPUT_DIR / "selection_manifest.json"
OUTPUT_PROTOCOL = OUTPUT_DIR / "SAMPLING_PROTOCOL.md"
OUTPUT_ZIP = Path(
    "/kaggle/working/mmmu_pro_heldout_500_artifacts.zip"
)


def sha256_text_lines(lines, sort_lines=False):
    values = sorted(lines) if sort_lines else list(lines)
    payload = ("\n".join(values) + "\n").encode("utf-8")
    return sha256(payload).hexdigest()


ordered_id_list = selected["id"].tolist()

with OUTPUT_IDS.open("w", encoding="utf-8") as f:
    f.write("\n".join(ordered_id_list) + "\n")

sorted_id_list = sorted(ordered_id_list)
with OUTPUT_IDS_SORTED.open("w", encoding="utf-8") as f:
    f.write("\n".join(sorted_id_list) + "\n")

selected_export = selected[
    [
        "selection_order",
        "dataset_index",
        "id",
        "subject",
        "topic_difficulty",
    ]
].copy()
selected_export.to_csv(OUTPUT_METADATA, index=False)

subject_by_difficulty.to_csv(OUTPUT_DISTRIBUTION)
candidate_by_subject_difficulty.to_csv(OUTPUT_CANDIDATES)
allocation_df.to_csv(OUTPUT_ALLOCATION_AUDIT, index=False)

excluded[
    [
        "dataset_index",
        "id",
        "subject",
        "topic_difficulty",
        "exclude_dev_id",
        "exclude_demo_id",
        "exclude_dev_exact_text",
        "exclude_demo_exact_text",
    ]
].to_csv(OUTPUT_EXCLUSION_AUDIT, index=False)


def pkg_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": (
        "Independent held-out cohort for adaptive prompt-routing evaluation"
    ),
    "dataset": {
        "repository": DATASET_REPO,
        "configuration": DATASET_CONFIG,
        "split": DATASET_SPLIT,
        "revision": DATASET_REVISION,
        "expected_rows": EXPECTED_MMMU_PRO_N,
        "observed_rows": len(metadata),
    },
    "sampling": {
        "seed": SEED,
        "target_n": TARGET_N,
        "subjects_observed": int(metadata["subject"].nunique()),
        "subject_allocation_method": (
            "seeded balanced round-robin with capacity constraints"
        ),
        "difficulty_target_proportions": TARGET_DIFFICULTY_PROPORTIONS,
        "difficulty_allocation_method": (
            "closest feasible integer allocation minimizing squared "
            "deviation from target proportions"
        ),
        "replacement": False,
        "performance_dependent_selection": False,
        "answer_labels_used_for_selection": False,
    },
    "exclusion_sources": {
        "development_id_file": str(SELECTED_IDS_PATH),
        "development_unique_ids": len(development_ids),
        "development_file_sha256": sha256_file(SELECTED_IDS_PATH),
        "demonstration_csv": str(DEMO_CSV_PATH),
        "demonstration_unique_ids": len(demonstration_ids),
        "demonstration_unique_normalized_questions": len(
            demonstration_questions
        ),
        "demonstration_csv_sha256": sha256_file(DEMO_CSV_PATH),
        "id_overlap_between_exclusion_sources": len(
            development_ids & demonstration_ids
        ),
    },
    "pool_audit": {
        "development_ids_found_in_mmmu_pro": int(
            metadata["exclude_dev_id"].sum()
        ),
        "demo_ids_found_in_mmmu_pro": int(
            metadata["exclude_demo_id"].sum()
        ),
        "rows_matching_development_exact_text": int(
            metadata["exclude_dev_exact_text"].sum()
        ),
        "rows_matching_demo_exact_text": int(
            metadata["exclude_demo_exact_text"].sum()
        ),
        "unique_rows_excluded": len(excluded),
        "eligible_candidate_rows": len(candidates),
    },
    "heldout_cohort": {
        "rows": len(selected),
        "unique_ids": int(selected["id"].nunique()),
        "ordered_id_sha256": sha256_text_lines(
            ordered_id_list,
            sort_lines=False,
        ),
        "set_id_sha256_sorted": sha256_text_lines(
            ordered_id_list,
            sort_lines=True,
        ),
        "subject_counts": {
            str(k): int(v)
            for k, v in (
                selected["subject"]
                .value_counts()
                .sort_index()
                .items()
            )
        },
        "difficulty_counts": {
            str(k): int(
                (selected["topic_difficulty"] == k).sum()
            )
            for k in DIFFICULTY_ORDER
        },
    },
    "integrity_checks": {
        name: bool(value)
        for name, value in checks.items()
    },
    "software": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "datasets": pkg_version("datasets"),
        "huggingface_hub": pkg_version("huggingface_hub"),
        "pyarrow": pkg_version("pyarrow"),
    },
}

with OUTPUT_MANIFEST.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

protocol_text = f"""# MMMU-Pro Held-Out Cohort Sampling Protocol

- Dataset: `{DATASET_REPO}`
- Configuration: `{DATASET_CONFIG}`
- Split: `{DATASET_SPLIT}`
- Revision: `{DATASET_REVISION}`
- Original dataset size: {len(metadata)}
- Development IDs excluded: {len(development_ids)}
- Demonstration IDs supplied: {len(demonstration_ids)}
- Unique MMMU-Pro rows excluded after all ID/text guards: {len(excluded)}
- Eligible pool after exclusion: {len(candidates)}
- Final held-out cohort: {len(selected)}
- Random seed: {SEED}
- Replacement: no
- Subject allocation: seeded balanced round-robin with capacity constraints
- Difficulty target: 30% Easy / 40% Medium / 30% Hard within subject
- Difficulty fallback: closest feasible integer allocation
- Answer labels used for selection: no
- Prompt/model performance used for selection: no
- ID overlap with development cohort: 0
- ID overlap with demonstration bank: 0
- Exact normalized-text overlap with development cohort: 0
- Exact normalized-text overlap with demonstration bank: 0
- Ordered cohort SHA-256: {manifest["heldout_cohort"]["ordered_id_sha256"]}
- Canonical set SHA-256: {manifest["heldout_cohort"]["set_id_sha256_sorted"]}
"""

OUTPUT_PROTOCOL.write_text(protocol_text, encoding="utf-8")

artifact_files = [
    OUTPUT_IDS,
    OUTPUT_IDS_SORTED,
    OUTPUT_METADATA,
    OUTPUT_DISTRIBUTION,
    OUTPUT_CANDIDATES,
    OUTPUT_EXCLUSION_AUDIT,
    OUTPUT_ALLOCATION_AUDIT,
    OUTPUT_MANIFEST,
    OUTPUT_PROTOCOL,
]

with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file_path in artifact_files:
        zf.write(file_path, arcname=file_path.name)

print("=" * 78)
print("EXPORT COMPLETE")
print("=" * 78)
for path in artifact_files:
    print(path)

print("\nZIP bundle:")
print(OUTPUT_ZIP)

EXPORT COMPLETE
/kaggle/working/mmmu_pro_heldout_500/selected_500_ids.txt
/kaggle/working/mmmu_pro_heldout_500/selected_500_ids_sorted.txt
/kaggle/working/mmmu_pro_heldout_500/selected_500_metadata.csv
/kaggle/working/mmmu_pro_heldout_500/subject_difficulty_distribution.csv
/kaggle/working/mmmu_pro_heldout_500/candidate_subject_difficulty_distribution.csv
/kaggle/working/mmmu_pro_heldout_500/exclusion_audit.csv
/kaggle/working/mmmu_pro_heldout_500/allocation_audit.csv
/kaggle/working/mmmu_pro_heldout_500/selection_manifest.json
/kaggle/working/mmmu_pro_heldout_500/SAMPLING_PROTOCOL.md

ZIP bundle:
/kaggle/working/mmmu_pro_heldout_500_artifacts.zip


## 14. Final frozen-cohort verification

Run this cell **after export**. It re-reads the saved ID file rather than trusting the in-memory object and verifies that the artifact itself contains exactly the intended 500 unique, disjoint IDs.

After this cell passes, the cohort should be treated as **frozen**. Do not regenerate it in response to model performance on these 500 items.

In [13]:
saved_ids = [
    normalize_id(line)
    for line in OUTPUT_IDS.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

final_artifact_checks = {
    "saved_row_count_is_500": len(saved_ids) == TARGET_N,
    "saved_ids_are_unique": len(set(saved_ids)) == TARGET_N,
    "saved_equals_in_memory_order": saved_ids == ordered_id_list,
    "saved_no_dev_id_overlap": len(set(saved_ids) & development_ids) == 0,
    "saved_no_demo_id_overlap": len(set(saved_ids) & demonstration_ids) == 0,
    "saved_hash_matches_manifest": (
        sha256_text_lines(saved_ids, sort_lines=False)
        == manifest["heldout_cohort"]["ordered_id_sha256"]
    ),
}

final_check_df = pd.DataFrame(
    [
        {"check": name, "passed": bool(value)}
        for name, value in final_artifact_checks.items()
    ]
)
display(final_check_df)

if not final_check_df["passed"].all():
    failed = final_check_df.loc[
        ~final_check_df["passed"],
        "check",
    ].tolist()
    raise AssertionError(
        f"Final artifact verification failed: {failed}"
    )

print(
    "\nSUCCESS: the 500-sample held-out cohort is frozen, "
    "unique, reproducible, and leakage-checked."
)
print(
    "Canonical cohort SHA-256:",
    manifest["heldout_cohort"]["set_id_sha256_sorted"],
)

,check,passed
0,saved_row_count_is_500,True
1,saved_ids_are_unique,True
2,saved_equals_in_memory_order,True
3,saved_no_dev_id_overlap,True
4,saved_no_demo_id_overlap,True
5,saved_hash_matches_manifest,True



SUCCESS: the 500-sample held-out cohort is frozen, unique, reproducible, and leakage-checked.
Canonical cohort SHA-256: 4a897112568e1cf5c9be6b831407a94d9e0e807a30dc283924c23877777451d4


## 15. Methods paragraph for thesis / paper

> **Held-out cohort construction.** A separate 500-instance evaluation cohort was sampled from the MMMU-Pro Standard (10 options) benchmark using a pinned dataset revision. Before sampling, all instances used in the 300-instance prompt-development cohort were excluded by identifier. Instances appearing in the demonstration/CoT bank were also excluded. As a conservative contamination safeguard, rows with exactly matching question text after Unicode normalization, whitespace normalization, and case folding were excluded as well. No fuzzy similarity criterion, answer label, model output, or prompt-performance statistic was used during test-set construction. The remaining pool was sampled without replacement using a fixed random seed. Samples were allocated as evenly as possible across the 30 academic subjects, while the within-subject allocation targeted a 30%/40%/30% Easy/Medium/Hard distribution. When a difficulty stratum lacked sufficient eligible instances, the closest feasible integer allocation to the target distribution was used and recorded in the audit log. The final cohort was then frozen prior to prompt-router evaluation.

### Interpretation

This notebook establishes **instance-level and exact-text disjointness**. It does not claim semantic de-duplication of visually or conceptually similar questions. Such semantic filtering is intentionally avoided because it would require an additional similarity metric and threshold that could itself alter the held-out sampling distribution.